# Artificial Intelligence in Games
## Project - Modelling Racing Cars using Machine Learning



## Introduction
In previous lab classes we have shown a pygame car race where we have manually controlled cars and also autonomous ones. In the autonomous we have implemented cars that are controlled by decision trees. The car that is controlled by decision trees presented in lab9 have 5 sensors which measure if the sensor position is on the road our outside of it (border or grass). These 5 sensors have fixed positions and are limited because they do not adapt to the road pattern. Cars are controlled by discrete actions like rotateLeft, rotateRight and move

## Goal

In this project the idea is to model some cars, manually or autonomous ones, in different situations using machine learning techniques, namely machine learning classifiers. In order to model it is necessary to gather data from different races with the same car and learn a classifier like a decision tree or other that best reproduce the car behavior.

## Specifications
This project is open to some variation and experimentation but there are some requisites we suggest to be considered

### Radars
<img src="figs/radarsCar.png" alt="Drawing" style="width: 300px;"/>
In contrast with the cars used in Lab9 that have fixed sizes and are a kind of color sensor that detect 3 types of materials, i.e., pixels: road, edge and grass, cars should be have radars that measure the distance towards the road edge. You might use 5 or more radars and each one has a maximum distace of range like in the figure above.
Note that in clear contrast with the discrete sensors used in Lab9 shich possess discrete values (a range of 3 possible values) these radar sensors that measure ditance towards the grass will have float values

### Noise
You should add some noise in the actions so that the rotations and movements possess a certain uncertainty.

### Modelled Cars
The modelled cars might be manually controlled or autonomous. We suggest the use of both types of cars to serve as cars to be modeled by classifiers.


### Different Races
To model a certain car, you need to automatically gather data from a set of race car performances in different situations that will be used in the leanring process. You do not need to change the circuit, but using the same circuit there is the possibility of same variation. For example, we can vary the initial directions of the cars in the sterting position or the position of the finish line and the race direction, that can be clockwise or anti-clockwise.

### Classifiers
You should use at leat the decision tree classifier in Scikitlearn library but you are free to test other classifiers. Remember that are always some parameters that we can vary in each classifier. For example, in decision trees we can use pruning and there are different tree pruning techniques and parameters that might be used. You can use other classifier, for example nave bayes ones that can be useful in case you use the original fixed size color sensors with manually controlled cars.

### Evaluation
In order to evalutate differente classifiers and optimize their parameters you need to use standard evaluation technques like separation in test and train data and use of stratified cross validation or some other well jsutified evaluation procedure. You should also choose some measure for choosing the best classifier

### Peformance compairison
After choosing the best classifier it would be useful to cpmapre the path followed by the orignal car and the learned one in a certain circuit o a ser of circuits. For this you might calculate the squared sum of the error for example or some other measure where the distance between the 2 pairwise  localization points in the same instants corresponds to the error. 

## Resources
You might use for resolurces the lab8 and lab9 jupyter notebooks and also 3 noteboosk about learning classifiers that are in the project folder.

## Report
You must have a report that:
* Identify the group elements by their names and numbers   
* Sumarizes the work done
* Describe the data gathering process to be used in the machine learning process. How can we choose a certain car and how we can vary the circuit and start the data gathering and save the data in order to be used later for learning a classifier.
* Describe what was done to gather data in the project.
* Describe the machine learning work dobe: the classofiers tested, the hyper-paramaters that were used, and how were the different resulting classifiers evaluated and indicate the evaluated results and the best classifier. ShoW how they were compared with the original and also the measures used to compare them. Describe also how can we call the function to compare them.   

## Submisson
all the material, code, resources and material should be putted in a folder, zipped and submitted in the course Moodle page. The zip file must have a name that has this patter `IIAGamesProject_XX` where XX identifies the group number or id.

## Deadline
The zip file must be submitted until the following deadline: 30th of May, 1m before midnight.

# Project Start

### Decision Trees Classes

In [11]:
from decisionTrees import *
from sklearn.tree import DecisionTreeClassifier
import joblib  # or pickle if you used that

model = joblib.load("trained_tree_model.pkl")

### Game Setup

In [12]:
import numpy
import pygame
import math
import random 
import time
import csv
from utils import *
#from random import *
import sys


# The grass background auemntado 2.5 vezes mais
GRASS = scale_image(pygame.image.load("imgs/grass.jpg"), 2.5)
# The track image diminuida em 90%
TRACK = scale_image(pygame.image.load("imgs/track.png"), 0.9)

TRACK_MASK = scale_image(pygame.image.load("imgs/track-mask_refined.png"), 0.9)

# The track border image for collision detection diminuída em 90%
TRACK_BORDER = scale_image(pygame.image.load("imgs/track-border.png"), 0.9)
TRACK_BORDER_MASK = pygame.mask.from_surface(TRACK_BORDER)

# import cars images diminuídas em 55%
RED_CAR = scale_image(pygame.image.load("imgs/red-car.png"), 0.55)
GREEN_CAR = scale_image(pygame.image.load("imgs/green-car.png"), 0.55)
car_width,car_height=GREEN_CAR.get_size()

### half car-width and half car-height
HALF_WIDTH=car_width/2
HALF_HEIGHT=car_height/2
CAR_SIZE=HALF_WIDTH,HALF_HEIGHT

# Get the size of the track image
WIDTH, HEIGHT = TRACK.get_width(), TRACK.get_height()

# the window has the size of the track image
WIN = pygame.display.set_mode((WIDTH, HEIGHT))

# the name of the window
pygame.display.set_caption("Racing Game!")

pygame.font.init()
MAIN_FONT = pygame.font.SysFont("comicsans", 44)

FPS = 30

RED = (255, 0, 0, 255)
WHITE = (255, 255, 255, 255)
YELLOW = (255, 255, 0, 255)


# The finish image intacta
horizontal_finish = pygame.image.load("imgs/finish.png")
vertical_finish = pygame.transform.rotate(horizontal_finish,90) # Negative angle amounts will rotate clockwise.
diagonal1_finish = pygame.transform.rotate(horizontal_finish,45) # Negative angle amounts will rotate clockwise.
diagonal2_finish = pygame.transform.rotate(horizontal_finish,-135) # Negative angle amounts will rotate clockwise.

FINISH_MASK_horizontal = pygame.mask.from_surface(horizontal_finish)
FINISH_MASK_vertical = pygame.mask.from_surface(vertical_finish)
FINISH_MASK_diagonal1 = pygame.mask.from_surface(diagonal1_finish)
FINISH_MASK_diagonal2 = pygame.mask.from_surface(diagonal2_finish)


# Finish line options: [((image, mask), position)]
finish_line_options = [
    ((horizontal_finish, FINISH_MASK_horizontal), (130, 250)),
    ((horizontal_finish, FINISH_MASK_horizontal), (680, 580)),
    ((vertical_finish, FINISH_MASK_vertical), (550, 35)),
    ((vertical_finish, FINISH_MASK_vertical), (530, 220)),
    ((diagonal1_finish, FINISH_MASK_diagonal1), (170, 600)),
    ((diagonal2_finish, FINISH_MASK_diagonal2), (100, 520))]


random_car_location = [((170,200),0),((730,530), 0),((510,55) , 90),((570,240), -90),
    ((170,585), 45),((160,565), -135)]

def random_track_func():
    finish_index = random.randint(0, len(finish_line_options) - 1)

    (finish_img, finish_mask), finish_pos = finish_line_options[finish_index]
    start_pos, start_angle = random_car_location[finish_index]

    images = [
        (GRASS, (0, 0)),
        (TRACK, (0, 0)),
        (finish_img, finish_pos),
        (TRACK_BORDER, (0, 0)),
    ]

    return images, finish_mask, finish_pos, start_pos, start_angle

In [13]:
#def draw(win,images,player_car,computer_car,game_info):
def draw(win,images,computer_car,game_info):

    for img,pos in images:
        win.blit(img,pos)
    level_text = MAIN_FONT.render(f'Level {game_info.level}',1,(255,255,255))
    win.blit(level_text,(10,HEIGHT-level_text.get_height()-90))
    
    time_text = MAIN_FONT.render(f'Time {game_info.get_level_time()}',1,(255,255,255))
    win.blit(time_text,(10,HEIGHT-time_text.get_height()-50))
    
    velocity_text = MAIN_FONT.render(f'Vel {round(computer_car.vel,1)} px/s',1,(255,255,255))
    win.blit(velocity_text,(10,HEIGHT-velocity_text.get_height()-10))
    
    computer_car.draw(win)
    pygame.display.update()

### Game Classes

In [14]:
class GameInfo:
    LEVELS = 10

    def __init__(self, level=1):
        self.level = level
        self.started = False
        self.level_start_time = 0

    def randomize_track(self):
        finish_index = random.randint(0, len(finish_line_options) - 1)
        (finish_img, finish_mask), finish_pos = finish_line_options[finish_index]
        images = [
            (GRASS, (0, 0)),
            (TRACK, (0, 0)),
            (finish_img, finish_pos),
            (TRACK_BORDER, (0, 0))]
        
        start_pos, angle = random_car_location[finish_index]
        return images, start_pos, angle, finish_mask, finish_pos

    def next_level(self):
        self.level += 1
        self.started = False

    def reset(self):
        self.level = 1
        self.started = False
        self.level_start_time = 0
        return self.randomize_track()
    
    def game_finished(self):
        return self.level > self.LEVELS

    def start_level(self):
        self.started = True
        self.level_start_time = time.time()

    def get_level_time(self):
        if not self.started:
            return 0
        return round(time.time() - self.level_start_time)

In [15]:
class AbstractCar():
    
    def __init__(self, max_vel, rotation_vel):
        self.img = self.IMG
        self.max_vel = max_vel
        self.vel = 0
        self.rotation_vel = rotation_vel
        self.angle = self.start_angle # era 0
        self.x,self.y = self.start_pos
        self.acceleration=0.1

       
    def rotate(self, left=False, right=False):
        if left and right:
            pass
        elif left:
            self.angle += self.rotation_vel + random.randrange(-1,1)
        elif right:
            self.angle -= self.rotation_vel +  random.randrange(-1,1)
        self.angle = int(self.angle) % 360
        
    def move_forward(self):
        self.vel = min(self.vel + self.acceleration, self.max_vel)
        self.move()
        
    def move_backwards(self):
        self.vel = max(self.vel - self.acceleration, -self.max_vel//2)
        self.move()
    
    
    # primeiro calculamos o  ângulo em radianos
    # Calculamos o deslocamento em x e y através da trignometra
    # actualizamos x e y mas subtraindo devido aos pontos cardeais do pygame e da corrida de carros
    def move(self):
        radians = math.radians(self.angle)
        vertical = math.cos(radians) * self.vel
        horizontal = math.sin(radians) * self.vel

        self.y -= vertical
        self.x -= horizontal
        
    
    def collide(self,mask,x=0,y=0):
            car_mask=pygame.mask.from_surface(self.img)
            offset=(int(self.x-x),int(self.y-y))
            poi=mask.overlap(car_mask,offset)
            return poi
        
    def reset(self):
        self.x,self.y=self.START_POS
        self.angle=0
        self.vel=0

    def draw(self, win):
        blit_rotate_center(win, self.img, (self.x, self.y), self.angle)
        x, y = int(self.x + CAR_SIZE[0]), int(self.y + CAR_SIZE[1])
        dx, dy = CAR_SIZE[0], CAR_SIZE[1]
        alfa = (self.angle) * math.pi / 180
        mrot = numpy.array([[math.cos(alfa), -math.sin(alfa)], [math.sin(alfa), math.cos(alfa)]])
        pts = numpy.array([[-dx, -dy], [dx, -dy], [-dx, dy], [dx, dy]])
        npts = numpy.dot(pts, mrot)
        for i in npts:
            if 0<=i[0]+x<WIDTH and 0<=i[1]+y<HEIGHT: 
                pygame.draw.circle(win, TRACK_MASK.get_at((int(i[0] + x), int(i[1] + y))), \
                                                          (int(i[0] + x), int(i[1] + y)),2, 2)

In [16]:
def handle_collision(player_car,game_info):
    if player_car.collide(TRACK_BORDER_MASK) != None:
        player_car.bounce()
        
    player_finish_poi_collide = player_car.collide(finish_mask, *finish_pos)
    
    if player_finish_poi_collide != None:
        if player_finish_poi_collide[1] == 0:
            player_car.bounce()
        else:
            player_car.reset()
            game_info.next_level()
            return True
    return False

### Decision Trees

#### Smoother Tree 4 sensors

acc = Action("accelerate")
brake = Action("brake")
rotate_left = Action("rotLeft")
rotate_right = Action("rotRight")

avoid_crash = Less("north_east?", rotate_left, brake, 20)

steer_left = Less("north_east?", rotate_left, acc, 45)

north_west_avoid_crash = Less("north_west?", rotate_right, avoid_crash, 20)

north_west_steer = Less("north_west?", rotate_right, steer_left, 45)

north_check = Less("north?", north_west_avoid_crash, north_west_steer, 30)


speed_check = Less("south?", brake, acc, 20)  # If something is very close behind, stop

final_tree = Less("north?", speed_check, north_check, 30)

decisionTreeSimple = final_tree

### Cluncky 6 sensors

acc = Action("accelerate")
brake = Action("brake")
rotate_left = Action("rotLeft")
rotate_right = Action("rotRight")

avoid_crash = Less("north_east?", rotate_left, brake, 20)

steer_left = Less("north_east?", rotate_left, acc, 45)

north_west_avoid_crash = Less("north_west?", rotate_right, avoid_crash, 20)

north_west_steer = Less("north_west?", rotate_right, steer_left, 45)

north_check = Less("north?", north_west_avoid_crash, north_west_steer, 30)


speed_check = Less("south?", brake, acc, 20)  # If something is very close behind, stop

north = Less("north?", speed_check, north_check, 30)

#### Side alignment (only if forward is clear)
side_align = Less("west?", rotate_right,
                  Less("east?", rotate_left, north, 5), 5)

#### Final: brake if something is behind
final_tree = Less("south?", brake, side_align, 10)

decisionTreeSimple = final_tree


In [17]:
class DTSimpleCar(AbstractCar):
    IMG = GREEN_CAR
    START_POS = (170, 200)
    SENSOR_POS = [(0, -50),(-30, -40),(30, -40),(0,20),(-50,-5),(50,-5)]
    SENSOR_NAMES = ['north','north_west','north_east','south','west','east']
    #DT = decisionTreeSimple

    def __init__(self, max_vel, rotation_vel, start_pos, start_angle, MAX_RANGE = 100): 
        self.max_range = MAX_RANGE  # Maximum radar distance
        self.start_pos = start_pos
        self.start_angle = start_angle
        
        super().__init__(max_vel, rotation_vel)
        self.update_sensors()
        self.x, self.y = start_pos
        self.angle = start_angle
        # deveria ser feito o UPDATE_SENSORS para termos os primeiros valores


    def step(self):
        self.update_sensors()
        valores = {}
    
        #for i, name in enumerate(self.SENSOR_NAMES):
            #valores[name + "?"] = self.sensors[i]

        #action = self.DT.decide(valores)
        #eval('self.' + action + '()')

    
    def next_level(self,level):
        self.reset()
        self.vel=self.max_vel+(level-1)*0.02
        
    def draww_sensors(self, win):
        for (pos,color) in self.sensors:
            if color:
                pygame.draw.circle(win, color,pos,2, 2)
                
    def draw(self,win):
        super().draw(win)
        self.draw_sensors(win)
        

    def update_sensors(self):
        self.sensors = []
        car_x = int(self.x + CAR_SIZE[0])
        car_y = int(self.y + CAR_SIZE[1])
        
        angle_rad = math.radians(self.angle)
    
        # Convert relative SENSOR_POS to actual radar directions
        mrot = numpy.array([
            [math.cos(angle_rad), -math.sin(angle_rad)],
            [math.sin(angle_rad),  math.cos(angle_rad)]
        ])
        sensor_dirs = numpy.dot(numpy.array(self.SENSOR_POS), mrot)
        
        for dir_x, dir_y in sensor_dirs:
            norm = math.sqrt(dir_x**2 + dir_y**2)
            if norm == 0: continue
            dir_x /= norm
            dir_y /= norm
    
            distance = 0
            for i in range(self.max_range):
                sx = int(car_x + dir_x * i)
                sy = int(car_y + dir_y * i)
                if 0 <= sx < WIDTH and 0 <= sy < HEIGHT:
                    color = TRACK_MASK.get_at((sx, sy))
                    if color != WHITE:  # Replace with your actual road color
                        break
                    distance += 1
                else:
                    break
            self.sensors.append(distance)  # Only store distance now

        # In your PlayerCar class or main loop:
        sensor_data = [self.sensors[i] for i in range(len(self.SENSOR_NAMES))]
        action = model.predict([sensor_data])[0]  # Get predicted action like "accelerate", "brake", etc.
        
        # Convert to actual action
        if action == "accelerate":
            self.accelerate()
        elif action == "brake":
            self.brake()
        elif action == "rotLeft":
            self.rotLeft()
        elif action == "rotRight":
            self.rotRight()
        elif action == 'accelerate+rotRight':
            self.accelerate()
            self.rotRight()
        elif action == 'accelerate+rotLeft':
            self.accelerate()
            self.rotLeft()

    def draw_sensors(self, win):
        car_x = int(self.x + CAR_SIZE[0])
        car_y = int(self.y + CAR_SIZE[1])
        
        angle_rad = math.radians(self.angle)
        mrot = numpy.array([
            [math.cos(angle_rad), -math.sin(angle_rad)],
            [math.sin(angle_rad),  math.cos(angle_rad)]
        ])
        sensor_dirs = numpy.dot(numpy.array(self.SENSOR_POS), mrot)
    
        for (dx, dy), dist in zip(sensor_dirs, self.sensors):
            norm = math.sqrt(dx**2 + dy**2)
            dx /= norm
            dy /= norm
            end_x = int(car_x + dx * dist)
            end_y = int(car_y + dy * dist)
            pygame.draw.line(win, (0, 255, 0), (car_x, car_y), (end_x, end_y), 1)
            pygame.draw.circle(win, (255, 255, 255), (end_x, end_y), 2)
                
     # Atomic Actions
    def rotLeft(self):
        self.rotate(True, False)

    def rotRight(self):
        self.rotate(False, True)

    def accelerate(self):
        self.move_forward()

    def brake(self):
        self.move_backwards()

    def bounce(self):
        self.vel=-self.vel
        self.move()

### Run Game

In [22]:
#FINISH_POSITION = (130,250)
#Car START_POS = (170, 200)
#player_car = PlayerCar(4,4)

images, finish_mask, finish_pos, start_pos, angle = random_track_func()
computer_car = DTSimpleCar(4,4,start_pos, angle)

game_info = GameInfo()
run = True
clock = pygame.time.Clock()

while run:
    clock.tick(FPS)
    #draw(WIN,images,player_car,computer_car,game_info)
    draw(WIN,images,computer_car,game_info)

    while not game_info.started:
        blit_text_center(WIN, MAIN_FONT, f'Press any key to start level {game_info.level}!')
        pygame.display.update()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                run=False
                pygame.quit()
                sys.exit()
            if event.type==pygame.KEYDOWN:
                game_info.start_level()
            
    
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            run=False
            break
                            
    #move_player(player_car)
    computer_car.step()
    
    #if handle_collision(player_car, computer_car,game_info):
        #draw(WIN,images,player_car,computer_car,game_info)

    if handle_collision(computer_car ,game_info):
        draw(WIN,images,computer_car,game_info)
    
    if game_info.game_finished():
        blit_text_center(WIN,MAIN_FONT,"YOU WON!")
        pygame.time.wait(5000)
        game_info.reset()
        player_car.start_pos = start_pos
        player_car.start_angle = angle
        player_car.reset()
        
#pygame.quit()


### EXTRA

In [10]:
   
''' 
def handle_collision(player_car, computer_car,game_info):
    if player_car.collide(TRACK_BORDER_MASK) != None:
        player_car.bounce()

    computer_finish_poi_collide = computer_car.collide(
        FINISH_MASK, *FINISH_POSITION)
    if computer_finish_poi_collide != None:
        blit_text_center(WIN,MAIN_FONT,"YOU LOST!")
        pygame.display.update()
        pygame.time.wait(5000)
        game_info.reset()
        player_car.reset()
        computer_car.next_level(1)
        return True

    player_finish_poi_collide = player_car.collide(
        FINISH_MASK, *FINISH_POSITION)
    if player_finish_poi_collide != None:
        if player_finish_poi_collide[1] == 0:
            player_car.bounce()
        else:
            player_car.reset()
            game_info.next_level()
            computer_car.next_level(game_info.level)
            return True
    return False
'''

' \ndef handle_collision(player_car, computer_car,game_info):\n    if player_car.collide(TRACK_BORDER_MASK) != None:\n        player_car.bounce()\n\n    computer_finish_poi_collide = computer_car.collide(\n        FINISH_MASK, *FINISH_POSITION)\n    if computer_finish_poi_collide != None:\n        blit_text_center(WIN,MAIN_FONT,"YOU LOST!")\n        pygame.display.update()\n        pygame.time.wait(5000)\n        game_info.reset()\n        player_car.reset()\n        computer_car.next_level(1)\n        return True\n\n    player_finish_poi_collide = player_car.collide(\n        FINISH_MASK, *FINISH_POSITION)\n    if player_finish_poi_collide != None:\n        if player_finish_poi_collide[1] == 0:\n            player_car.bounce()\n        else:\n            player_car.reset()\n            game_info.next_level()\n            computer_car.next_level(game_info.level)\n            return True\n    return False\n'

In [ ]:
'''class PlayerCar(AbstractCar):
    IMG = RED_CAR
    START_POS = (130, 200)
    SENSOR_POS = [(0, -50), (-20, -30), (20, -30), (0, 40)]
    SENSOR_NAMES = ['north',"west","east","south"]

    
    #PATH=[(161, 148), (147, 82), (68, 93), (61, 174), (64, 251), (59, 464), (293, 711), (399, 712), (407, 537), (491, 474), (591, 529), (618, 707), (731, 709), (739, 383), (395, 334), (438, 250), (706, 251), (738, 96), (287, 93), (273, 395), (179, 399), (173, 258)]

    
    def __init__(self, max_vel, rotation_vel, MAX_RANGE = 100):
        self.max_range = MAX_RANGE  # Maximum radar distance
        self.log_file = open("training_data.csv", "w", newline="")
        self.logger = csv.writer(self.log_file)
        self.logger.writerow(["north?", "west?", "east?", "south?", "action"])
        
        super().__init__(max_vel, rotation_vel)
        self.update_sensors()


    def step(self):
        self.update_sensors()
        valores = {}
        row = [self.sensors[i] for i in range(len(self.SENSOR_NAMES))] + [self.last_action]
        self.logger.writerow(row)

    def reduce_speed(self):
        self.vel = max(self.vel - self.acceleration / 2, 0)
        self.move()
        
    def bounce(self):
        self.vel=-self.vel
        self.move()
    
        
    def draww_sensors(self, win):
        for (pos,color) in self.sensors:
            if color:
                pygame.draw.circle(win, color,pos,2, 2)
                
    def draw(self,win):
        super().draw(win)
        self.draw_sensors(win)
        

    def update_sensors(self):
        self.sensors = []
        car_x = int(self.x + CAR_SIZE[0])
        car_y = int(self.y + CAR_SIZE[1])
        
        angle_rad = math.radians(self.angle)
    
        # Convert relative SENSOR_POS to actual radar directions
        mrot = numpy.array([
            [math.cos(angle_rad), -math.sin(angle_rad)],
            [math.sin(angle_rad),  math.cos(angle_rad)]
        ])
        sensor_dirs = numpy.dot(numpy.array(self.SENSOR_POS), mrot)
        
        for dir_x, dir_y in sensor_dirs:
            norm = math.sqrt(dir_x**2 + dir_y**2)
            if norm == 0: continue
            dir_x /= norm
            dir_y /= norm
    
            distance = 0
            for i in range(self.max_range):
                sx = int(car_x + dir_x * i)
                sy = int(car_y + dir_y * i)
                if 0 <= sx < WIDTH and 0 <= sy < HEIGHT:
                    color = TRACK_MASK.get_at((sx, sy))
                    #print(color)
                    # Assuming road is GRAY; everything else is off-road
                    if color != WHITE:  # Replace with your actual road color
                        break
                    distance += 1
                else:
                    break
            self.sensors.append(distance)  # Only store distance now

    def draw_sensors(self, win):
        car_x = int(self.x + CAR_SIZE[0])
        car_y = int(self.y + CAR_SIZE[1])
        
        angle_rad = math.radians(self.angle)
        mrot = numpy.array([
            [math.cos(angle_rad), -math.sin(angle_rad)],
            [math.sin(angle_rad),  math.cos(angle_rad)]
        ])
        sensor_dirs = numpy.dot(numpy.array(self.SENSOR_POS), mrot)
    
        for (dx, dy), dist in zip(sensor_dirs, self.sensors):
            norm = math.sqrt(dx**2 + dy**2)
            dx /= norm
            dy /= norm
            end_x = int(car_x + dx * dist)
            end_y = int(car_y + dy * dist)
            pygame.draw.line(win, (0, 255, 0), (car_x, car_y), (end_x, end_y), 1)
            pygame.draw.circle(win, (255, 255, 255), (end_x, end_y), 2)
'''
